# **Database to Kaggle**
Run this script to transform a dataset for a Kaggle competition

## 1. Generate the synthetic dataset into a pandas DataFrame `db`

In [9]:
%pip install pandas
import pandas as pd

Note: you may need to restart the kernel to use updated packages.


In [19]:
import pandas as pd
import numpy as np

# ----------------------------
# CONFIGURACIÓN GENERAL
# ----------------------------
np.random.seed(42)
N = 100_000

# ----------------------------
# VARIABLES BASE
# ----------------------------
p_edad_meses = [0.01]*5 + [0.04]*24 + [0.01]*6  # distribución de edad
p_edad_meses = np.array(p_edad_meses) / np.sum(p_edad_meses)

df = pd.DataFrame({
    'edad_meses': np.random.choice(range(-5, 30), size=N, p=p_edad_meses),
    'sexo': np.random.choice(['M', 'F'], size=N),
    'region': np.random.choice(['Costa', 'Sierra', 'Amazonía', 'Insular'], size=N),
    'nivel_educacion_madre': np.random.choice(
        ['Sin escolaridad', 'Primaria', 'Secundaria', 'Superior'],
        size=N,
        p=[0.05, 0.35, 0.45, 0.15]
    )
})

# ----------------------------
# GENERACIÓN DE PESO Y TALLA REALISTAS
# ----------------------------
# Promedios y desviaciones para edad
df['peso_kg'] = np.random.normal(
    loc=3 + (df['edad_meses'].clip(0, 24) * 0.3),  # peso promedio según edad
    scale=1.0
)
df['talla_cm'] = np.random.normal(
    loc=50 + (df['edad_meses'].clip(0, 24) * 1.5),  # talla promedio según edad
    scale=3.5
)

# ----------------------------
# CÁLCULO DE UN ÍNDICE DE DESNUTRICIÓN SIMPLIFICADO
# ----------------------------
# índice: peso/(edad*factor) y talla/(edad*factor)
# niños con valores bajos tendrán mayor probabilidad de alerta
peso_edad_ratio = df['peso_kg'] / (df['edad_meses'].clip(1, 24))
talla_edad_ratio = df['talla_cm'] / (df['edad_meses'].clip(1, 24))

# ----------------------------
# FUNCIÓN LOGÍSTICA PARA PROBABILIDAD DE ALERTA
# ----------------------------
# factores de riesgo:
# - bajo peso relativo
# - baja talla relativa
# - madre sin educación
# - región Amazonía tiene ligeramente mayor riesgo
# - edades extremas (recién nacidos o >24 meses inconsistentes)
risk = (
    (-5 * (peso_edad_ratio < 0.25)) +        # bajo peso
    (-3 * (talla_edad_ratio < 3)) +          # baja talla
    (-2 * (df['nivel_educacion_madre'] == 'Sin escolaridad')) +
    (-1 * (df['region'] == 'Amazonía')) +
    (-1.5 * (df['edad_meses'] < 0))          # inconsistente
)

# Convertir riesgo a probabilidad (sigmoide)
prob_alerta = 1 / (1 + np.exp(-(risk + np.random.normal(0, 1, N))))

# ----------------------------
# ASIGNAR ALERTAS BASADAS EN PROBABILIDAD
# ----------------------------
df['alerta_desnutricion'] = (prob_alerta > np.quantile(prob_alerta, 0.9)).astype(int)
# (Aprox. 10% alertas, pero condicionadas al riesgo real)

# ----------------------------
# AÑADIR RUIDO Y ANOMALÍAS
# ----------------------------
# 10% faltantes en talla, 5% en peso
df.loc[df.sample(frac=0.1).index, 'talla_cm'] = np.nan
df.loc[df.sample(frac=0.05).index, 'peso_kg'] = np.nan

# 1% de pesos fuera de rango
outlier_idx = df.sample(frac=0.01).index
df.loc[outlier_idx, 'peso_kg'] = np.random.uniform(30, 60, len(outlier_idx))

# 1% de edades negativas
neg_age_idx = df.sample(frac=0.01).index
df.loc[neg_age_idx, 'edad_meses'] = np.random.randint(-12, 0, len(neg_age_idx))

# ----------------------------
# GUARDAR RESULTADO
# ----------------------------
df.to_csv('alertas_ninos_menores_2anos_sintetico.csv', index=False)

print(df['alerta_desnutricion'].value_counts(normalize=True))
df.head(10)


alerta_desnutricion
0    0.9
1    0.1
Name: proportion, dtype: float64


,edad_meses,sexo,region,nivel_educacion_madre,peso_kg,talla_cm,alerta_desnutricion
0,8,F,Amazonía,Primaria,5.317365,56.875374,0
1,24,F,Sierra,Secundaria,9.071434,86.308246,0
2,18,F,Sierra,Primaria,8.052828,76.698298,0
3,14,M,Insular,Secundaria,NaN,73.020969,0
4,2,M,Sierra,Primaria,3.108528,49.419534,0
5,2,M,Insular,Primaria,NaN,50.634298,0
6,0,F,Sierra,Secundaria,1.326684,44.269474,0
7,21,F,Insular,Superior,10.343559,84.530825,0
8,14,M,Insular,Secundaria,7.980937,75.375776,1
9,17,F,Insular,Sin escolaridad,9.944283,75.551930,0


In [11]:
df.shape

(100000, 7)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   edad_meses             100000 non-null  int64  
 1   sexo                   100000 non-null  object 
 2   region                 100000 non-null  object 
 3   nivel_educacion_madre  100000 non-null  object 
 4   peso_kg                95048 non-null   float64
 5   talla_cm               90000 non-null   float64
 6   alerta_desnutricion    100000 non-null  int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 5.3+ MB


In [13]:
df.describe()

,edad_meses,peso_kg,talla_cm,alerta_desnutricion
count,100000.000000,95048.000000,90000.000000,100000.000000
mean,11.465470,6.898304,67.464020,0.100000
std,8.252789,4.724715,11.934883,0.300002
min,-12.000000,-0.814205,34.904595,0.000000
25%,5.000000,4.500690,57.404873,0.000000
50%,12.000000,6.530321,67.419471,0.000000
75%,18.000000,8.555267,77.428739,0.000000
max,29.000000,59.976104,98.213648,1.000000


## 2. Isolate features (`X`) and target variable (`y`)

In [20]:
# Divide the dataset into features and target variable
X = df.drop('alerta_desnutricion', axis=1)
y = df['alerta_desnutricion']

## 3. Divide the dataset in training and testing sets

In [21]:
# Install scikit-learn if not already installed
%pip install scikit-learn

# Import the train_test_split function
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Note: you may need to restart the kernel to use updated packages.


## 4. Save the `X_train`, `y_train`, and `X_test` dataframes to `.csv` files

In [23]:
# Save the training and testing sets to CSV files
X_train.to_csv("X_train_Kaggle.csv", index=False)
X_test.to_csv("X_test_Kaggle.csv", index=False)
y_train.to_csv("y_train_Kaggle.csv", index=False)

## 5. Create the `solutionsIDClassUsage` dataframe

In [39]:
# Create a dataframe with the following columns:
# "id" with values from 1 to the length of y_test
# "class" with the values of y_test
# "usage" with 50% of the values "Public" and 50% "Private" randomized
import numpy as np
# Create a new DataFrame with the specified columns
solutionsIdClassUsage = pd.DataFrame({
    "ID": range(1, len(y_test) + 1),  # Generar IDs desde 1 hasta la longitud de y_test
    "alerta_desnutricion": y_test.values,  # Columna de clase con los valores de y_test
})
# Randomly assign "Public" and "Private" to the "usage" column
usage_values = np.random.choice(["Public", "Private"], size=len(solutionsIdClassUsage), p=[0.1, 0.9])
solutionsIdClassUsage["Usage"] = usage_values

# Display the first few rows of the new DataFrame
solutionsIdClassUsage.head(40)

,ID,alerta_desnutricion,Usage
0,1,0,Private
1,2,0,Private
2,3,0,Private
3,4,0,Private
4,5,0,Private
5,6,0,Private
6,7,0,Private
7,8,0,Private
8,9,0,Private
9,10,0,Private


In [40]:
# Export the DataFrame to a CSV file
solutionsIdClassUsage.to_csv("solutionsIdClassUsage.csv", index=False)

In [33]:
# Display the shape of the DataFrame
solutionsIdClassUsage.shape

(30000, 3)

## 6. Create the `sample_submission` dataframe

In [37]:
# Create the sample_submission file with "id" and "class" columns. The length of "id" is the same as that of y_test, and the "class" column should have all values ​​equal to 0.
sample_submission = pd.DataFrame({
    "ID": range(1, len(y_test) + 1),  # Generate IDs from 1 to the length of y_test
    "alerta_desnutricion": 0  # Column of class with all values equal to 0
})
# Show the first few rows of the sample_submission DataFrame
sample_submission.head()

,ID,alerta_desnutricion
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [38]:
# Save the sample_submission DataFrame to a CSV file
sample_submission.to_csv("sample_submission_malnutrition.csv", index=False)

In [36]:
# Display the shape of the DataFrame
sample_submission.shape

(30000, 2)